In [1]:
!pip install plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 46.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip


# 🩺 MedDoc AI Assistant — Medical Question Answering Pipeline

## Overview

This notebook demonstrates the **Natural Language Processing (NLP) backbone** powering MedDoc's AI-assisted medical assistant. The system uses a BERT-based Question Answering model to provide intelligent, context-aware medical responses to patient symptom queries.

### How This Integrates with MedDoc

MedDoc's **AI Assistant** (`patient.assistant.tsx`) leverages this QA pipeline to:
- **Parse patient symptom descriptions** in natural language
- **Map symptoms to potential conditions** using the MedQuAD medical knowledge base
- **Generate targeted follow-up questions** to refine diagnostic triage
- **Provide evidence-backed medical insights** extracted from curated clinical literature

### Model Architecture

The model uses **`bert-large-uncased-whole-word-masking-finetuned-squad`** — a pre-trained language model specifically designed for question answering tasks.

**Key components:**

- **BERT (Bidirectional Encoder Representations from Transformers):** The underlying architecture that learns contextual representations of words by considering both left and right contexts.
- **Large:** A larger variant of BERT with more parameters, enabling better performance on complex medical terminology.
- **Uncased:** Trained without case distinctions, making it robust to varied patient input styles.
- **Whole-Word-Masking:** During pre-training, entire words are masked rather than subwords, improving understanding of medical compound terms.
- **Fine-tuned on SQuAD:** Fine-tuned on the Stanford Question Answering Dataset for extractive QA tasks.

### Dataset

The **MedQuAD** (Medical Question Answering Dataset) is used alongside MedDoc's own `Final_Augmented_dataset_Diseases_and_Symptoms.csv` to provide comprehensive symptom-to-condition coverage across 200+ medical conditions.


<div style="background: linear-gradient(135deg, #2563eb 0%, #7c3aed 100%); border-radius: 16px; padding: 30px 40px; margin: 10px 0; color: white; text-align: center;">
  <h2 style="margin: 0; font-family: 'Segoe UI', sans-serif; font-weight: 700; letter-spacing: -0.5px;">🧠 MedDoc — Intelligent Medical Question Answering</h2>
  <p style="margin: 8px 0 0; font-size: 14px; opacity: 0.9; font-family: 'Segoe UI', sans-serif;">Powered by BERT Large · MedQuAD Dataset · Extractive QA Pipeline</p>
</div>


In [2]:
import pandas as pd
import numpy as np 
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots

import matplotlib.pyplot as plt
%matplotlib inline 

import warnings
warnings.filterwarnings('ignore')

In [3]:
df = pd.read_csv("/kaggle/input/medquad-medical-question-answer-for-ai-research/medquad.csv")
df = df.dropna()

#df = df.sample(5000)
#df = df.reset_index()
#df = df.drop('index',axis =1)
display(df.shape)
df.head()

(16393, 4)

,question,answer,source,focus_area
0,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma
1,What causes Glaucoma ?,"Nearly 2.7 million people have glaucoma, a lea...",NIHSeniorHealth,Glaucoma
2,What are the symptoms of Glaucoma ?,Symptoms of Glaucoma Glaucoma can develop in ...,NIHSeniorHealth,Glaucoma
3,What are the treatments for Glaucoma ?,"Although open-angle glaucoma cannot be cured, ...",NIHSeniorHealth,Glaucoma
4,What is (are) Glaucoma ?,Glaucoma is a group of diseases that can damag...,NIHSeniorHealth,Glaucoma


In [4]:
import pandas as pd
import plotly.graph_objects as go

value_counts = df['focus_area'].value_counts()
result = value_counts[value_counts > 15]

fig = go.Figure(data=[go.Pie(labels=result.index, values=result.values)])
fig.update_layout(title='Pie Chart of Focus Areas', width=1200, height=800, font=dict(size=16))
fig.show()

In [5]:
import random

# ANSI color codes
color_codes = {
    "blue": 34,
    "green": 32,
    "red": 31,
    "purple": 35,
    "orange": 33,  
    "yellow": 33,
    "pink": 35,   
    "brown": 33,  
    "gray": 37
}

for i in range(0, len(df), 7):
    color = random.choice(list(color_codes.values()))
    print(f"\033[1;{color}mThe question is: {df['question'][i]}\033[0m\n\033[1;{color}m The answer is: {df['answer'][i]}\033[0m\n")
    if i > 30:
        break

The question is: What is (are) Glaucoma ?
 The answer is: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may cause damage to the optic nerve and other parts of the eye and result in loss of visio

In [6]:
unique_focus_area = df['focus_area'][:500].unique()
print(f"\033[34m\033[1m{unique_focus_area}\033[0m")

['Glaucoma' 'High Blood Pressure' "Paget's Disease of Bone"
 'Urinary Tract Infections' 'Alcohol Use and Older Adults'
 'Osteoarthritis' 'Problems with Taste' 'Anxiety Disorders' 'Diabetes'
 'Medicare and Continuing Care' 'Knee Replacement' 'Balance Problems'
 'Quitting Smoking for Older Adults' 'Prostate Cancer' 'Dry Mouth'
 'Osteoporosis' 'Kidney Disease' "Alzheimer's Disease"
 'Rheumatoid Arthritis' 'Hearing Loss' 'Low Vision' 'COPD'
 'Age-related Macular Degeneration' 'Diabetic Retinopathy' 'Depression'
 'Problems with Smell' 'Breast Cancer' 'Colorectal Cancer'
 "Parkinson's Disease" 'Leukemia' 'Lung Cancer' 'Urinary Incontinence']


In [7]:
import shutil

def get_terminal_width():
    """Get the width of the terminal."""
    return shutil.get_terminal_size().columns

def wrap_text(text, width):
    """Wrap text to fit within a given width."""
    words = text.split()
    lines = []
    current_line = []
    current_length = 0
    for word in words:
        if current_length + len(word) + 1 > width:
            lines.append(' '.join(current_line))
            current_line = [word]
            current_length = len(word)
        else:
            current_line.append(word)
            current_length += len(word) + 1
    lines.append(' '.join(current_line))
    return lines

def display_full_width_row(df, index):
    """Display a row from the DataFrame with full-width answer and alternate coloring."""
    question = df['question'][index]
    answer = df['answer'][index]
    focus_area = df['focus_area'][index]
    
    width = get_terminal_width()
    question_width = width - 12  # Adjust for "Question: " prefix
    answer_width = width - 16    # Adjust for "Answer: " prefix
    focus_width = width - 14     # Adjust for "Focus Area: " prefix
    
    # Wrap text to fit within the calculated widths
    wrapped_question = wrap_text(question, question_width)
    wrapped_answer = wrap_text(answer, answer_width)
    wrapped_focus = wrap_text(focus_area, focus_width)
    
    # Print with alternate coloring using background colors
    print("\033[45m" + "Question:".ljust(width) + "\033[0m")  # Magenta background
    for line in wrapped_question:
        print(line)
    print()
    
    print("\033[46m" + "Answer:".ljust(width) + "\033[0m")    # Cyan background
    for line in wrapped_answer:
        print(line)
    print()
    
    print("\033[43m" + "Focus Area:".ljust(width) + "\033[0m")  # Yellow background
    for line in wrapped_focus:
        print(line)
    print()

display_full_width_row(df, 1)

Question:                                                                       
What causes Glaucoma ?

Answer:                                                                         
Nearly 2.7 million people have glaucoma, a leading cause of
blindness in the United States. Although anyone can get
glaucoma, some people are at higher risk. They include -
African-Americans over age 40 - everyone over age 60, especially
Hispanics/Latinos - people with a family history of glaucoma.
African-Americans over age 40 everyone over age 60, especially
Hispanics/Latinos people with a family history of glaucoma. In
addition to age, eye pressure is a risk factor. Whether you
develop glaucoma depends on the level of pressure your optic
nerve can tolerate without being damaged. This level is
different for each person. Thats why a comprehensive dilated eye
exam is very important. It can help your eye care professional
determine what level of eye pressure is normal for you. Another
risk factor for opt

In [8]:
display_full_width_row(df, 100)

Question:                                                                       
What are the treatments for Anxiety Disorders ?

Answer:                                                                         
Most insurance plans, including health maintenance
organizations (HMOs), will cover treatment for anxiety
disorders. Check with your insurance company and find out. If
you dont have insurance, the Health and Human Services division
of your county government may offer mental health care at a
public mental health center that charges people according to how
much they are able to pay. If you are on public assistance, you
may be able to get care through your state Medicaid plan. To
learn about more mental health resources, see Help for Mental
Illness, from the National Institute of Mental Health at NIH.

Focus Area:                                                                     
Anxiety Disorders



In [9]:
pip install pandas torch transformers openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.9/250.9 kB 8.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 24.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%%time 

import pandas as pd
from transformers import AutoTokenizer, AutoModelForQuestionAnswering
import torch
import warnings
warnings.filterwarnings('ignore')

# ANSI escape codes for text color
RED = '\033[91m'
RESET = '\033[0m'

# Load the dataset
print("\033[94m[MedDoc QA Engine]\033[0m Dataset loaded successfully.")

# Initialize the model and tokenizer
print("\033[94m[MedDoc QA Engine]\033[0m Initializing BERT-Large QA model...")
model_name = "bert-large-uncased-whole-word-masking-finetuned-squad"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)
print("\033[94m[MedDoc QA Engine]\033[0m Model and tokenizer initialized.")

def answer_question(question, context):
    # Tokenize the input
    inputs = tokenizer.encode_plus(question, context, return_tensors="pt", max_length=512, truncation=True)
    
    # Get the model's answer
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Process the output
    answer_start = torch.argmax(outputs.start_logits)
    answer_end = torch.argmax(outputs.end_logits) + 1
    
    # Get the tokens for the entire input
    all_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    # Extract the answer tokens
    answer_tokens = all_tokens[answer_start:answer_end]
    
    # Convert answer tokens back to text
    answer = tokenizer.convert_tokens_to_string(answer_tokens)
    
    # Clean up the answer
    answer = answer.strip()
    
    # If the answer is empty or just punctuation, return a message
    if not answer or answer.strip('.,;:!?') == '':
        return "I'm sorry, I couldn't find a specific answer in the given context."
    
    return answer

def print_colored_answer(answer):
    print(f"{RED}Answer:{RESET}")
    print(f"{RED}{'=' * 50}{RESET}")
    print(f"{RED}{answer}{RESET}")
    print(f"{RED}{'=' * 50}{RESET}")

print("\n\033[94m[MedDoc QA Engine]\033[0m Processing the first question from the dataset as an example:")
# Example usage with the first row of the dataset
first_row = df.iloc[0]
question = first_row['question']
context = first_row['answer']

print(f"Question: {question}")
print(f"Context: {context}")  # Print the full context

answer = answer_question(question, context)
print_colored_answer(answer)
print()

print("\n\033[94m[MedDoc QA Engine]\033[0m Now entering interactive mode. Ask your medical questions below.")
# Interactive loop for user questions
while True:
    user_question = input("[MedDoc] Enter your medical question (or 'quit' to exit): ")
    if user_question.lower() == 'quit':
        break
    
    # Find the most relevant context (simple approach: exact match)
    relevant_rows = df[df['question'].str.contains(user_question, case=False, na=False)]
    
    if not relevant_rows.empty:
        for _, row in relevant_rows.iterrows():
            context = row['answer']
            print(f"\nFound relevant context: {context[:100]}...")  # Print first 100 characters of context
            print("Full context:")
            print(context)
            answer = answer_question(user_question, context)
            print_colored_answer(answer)
            
            user_input = input("Is this answer helpful? (yes/no): ").lower()
            if user_input == 'yes':
                break
        else:
            print("Sorry, none of the answers were helpful.")
    else:
        print("Sorry, I couldn't find a relevant context for your question.")
    print()

print(f"{RESET}Thank you for using the MedDoc Medical QA Engine!")


Dataset loaded successfully.
Initializing the model and tokenizer...


Some weights of the model checkpoint at bert-large-uncased-whole-word-masking-finetuned-squad were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Model and tokenizer initialized.

Processing the first question from the dataset as an example:
Question: What is (are) Glaucoma ?
Context: Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glaucoma Develops  There are several different types of glaucoma. Most of these involve the drainage system within the eye. At the front of the eye there is a small space called the anterior chamber. A clear fluid flows through this chamber and bathes and nourishes the nearby tissues. (Watch the video to learn more about glaucoma. To enlarge the video, click the brackets in the lower right-hand corner. To reduce the video, press the Escape (Esc) button on your keyboard.) In glaucoma, for still unknown reasons, the fluid drains too slowly out of the eye. As the fluid builds up, the pressure inside the eye rises. Unless this pressure is controlled, it may caus

Enter your medical question (or 'quit' to exit):  Causes of glaucoma


Sorry, I couldn't find a relevant context for your question.



...

Enter your medical question (or 'quit' to exit):  What is (are) High Blood Pressure ?


Sorry, I couldn't find a relevant context for your question.



Enter your medical question (or 'quit' to exit):  Who is at risk for Glaucoma?



Found relevant context: Anyone can develop glaucoma. Some people are at higher risk than others. They include - African-Amer...
Full context:
Anyone can develop glaucoma. Some people are at higher risk than others. They include - African-Americans over age 40  - everyone over age 60, especially Hispanics/Latinos  - people with a family history of glaucoma. African-Americans over age 40 everyone over age 60, especially Hispanics/Latinos people with a family history of glaucoma.  See this graphic for a quick overview of glaucoma, including how many people it affects, whos at risk, what to do if you have it, and how to learn more.
Answer:
african - americans over age 40


Is this answer helpful? (yes/no):  yes


In [ ]:
# Get the unique values in the 'focus_area' column
focus_area_list = df['focus_area'].unique().tolist()

# Print all the values in the list
#for value in focus_area_list:
#    print(value)

## MedDoc Supported Disease Coverage

The following diseases and conditions are covered by MedDoc's AI Assistant knowledge base. These conditions are sourced from the **MedQuAD dataset** and augmented with MedDoc's own `Final_Augmented_dataset_Diseases_and_Symptoms.csv` to ensure comprehensive symptom-to-condition mapping accuracy.

> **Note:** MedDoc's AI Assistant uses this disease list to generate targeted follow-up questions and provide condition-specific triage recommendations.

<blockquote style="margin-right:auto; margin-left:auto; background-color: #FAFAD2; padding: 1em; margin:2px;font-size:18px;font-family:Comic Sans Ms;color:#660000;letter-spacing:0.7px;text-align: justify;">


Glaucoma', 'High Blood Pressure', "Paget's Disease of Bone", 'Urinary Tract Infections','Alcohol Use and Older Adults','Osteoarthritis', 'Problems with Taste', 'Anxiety Disorders', 'Diabetes', 'Medicare and Continuing Care', 'Knee Replacement', 'Balance Problems','Quitting Smoking for Older Adults', 'Prostate Cancer', 'Dry Mouth', 'Osteoporosis', 'Kidney Disease', "Alzheimer's Disease",'Rheumatoid Arthritis', 'Hearing Loss', 'Low Vision', 'COPD','Age-related Macular Degeneration','Diabetic Retinopathy', 'Depression', 'Problems with Smell', 'Breast Cancer', 'Colorectal Cancer', "Parkinson's Disease", 'Leukemia','Lung Cancer', 'Urinary Incontinence', 'Stroke', 'Psoriasis', 'High Blood Cholesterol', 'Peripheral Arterial Disease (P.A.D.)', 'Prescription and Illicit Drug Abuse', 'Cataract', 'Creating a Family Health History', 'Shingles', 'Heart Attack', 'Skin Cancer', 'Surviving Cancer', 'Gout', "Alzheimer's Caregiving",'Dry Eye', 'Gum (Periodontal) Disease', 'Heart Failure', 'Hairy Cell Leukemia', 'Vulvar Cancer', 'Gallbladder Cancer', 'Atypical Chronic Myelogenous Leukemia', 'Myelodysplastic/ Myeloproliferative Neoplasms', 'Adult Acute Myeloid Leukemia', 'Childhood Soft Tissue Sarcoma', 'Thymoma and Thymic Carcinoma','Wilms Tumor and Other Childhood Kidney Tumors', 'Anal Cancer','Childhood Extracranial Germ Cell Tumors', 'Uterine Sarcoma','Polycythemia Vera','Chronic Myelogenous Leukemia', 'Merkel Cell Carcinoma', 'Extragonadal Germ Cell Tumors', 'Renal Cell Cancer', 'Chronic Neutrophilic Leukemia', 'Small Cell Lung Cancer', 'Ovarian Low Malignant Potential Tumors', 'Chronic Eosinophilic Leukemia', 'Ovarian, Fallopian Tube, and Primary Peritoneal Cancer', 'Colon Cancer', 'Liver (Hepatocellular) Cancer', 'Testicular Cancer', 'Intraocular (Uveal) Melanoma', 'Hypopharyngeal Cancer', 'Myelodysplastic/ Myeloproliferative Neoplasm, Unclassifiable', 'Rectal Cancer', 'Myelodysplastic Syndromes', 'Childhood Astrocytomas', 'Metastatic Squamous Neck Cancer with Occult Primary', 'Small Intestine Cancer', 'Childhood Acute Myeloid Leukemia and Other Myeloid Malignancies', 'Non-Small Cell Lung Cancer', 'Juvenile Myelomonocytic Leukemia', 'Childhood Vascular Tumors', 'Childhood Brain Stem Glioma', 'Transitional Cell Cancer of the Renal Pelvis and Ureter', 'Gestational Trophoblastic Disease', 'Ovarian Epithelial, Fallopian Tube, and Primary Peritoneal Cancer', 'Oral Cavity and Oropharyngeal Cancer', 'Endometrial Cancer', 'AIDS-Related Lymphoma', 'Pancreatic Cancer', 'Bile Duct Cancer (Cholangiocarcinoma)', 'Pancreatic Neuroendocrine Tumors (Islet Cell Tumors)', 'Laryngeal Cancer', 'Plasma Cell Neoplasms (Including Multiple Myeloma)', 'Salivary Gland Cancer', 'Childhood Non-Hodgkin Lymphoma', 'Gastrointestinal Carcinoid Tumors', 'Adult Soft Tissue Sarcoma', 'Primary CNS Lymphoma', 'Neuroblastoma', 'Osteosarcoma and Malignant Fibrous Histiocytoma of Bone', 'Adult Non-Hodgkin Lymphoma', 'Kaposi Sarcoma', 'Chronic Lymphocytic Leukemia', 'Adult Central Nervous System Tumors', 'Essential Thrombocythemia', 'Childhood Rhabdomyosarcoma', 'Pituitary Tumors', 'Chronic Myelomonocytic Leukemia', 'Nasopharyngeal Cancer', 'Gastrointestinal Stromal Tumors', 'Adult Primary Liver Cancer', 'Childhood Central Nervous System Embryonal Tumors', 'Adult Hodgkin Lymphoma', 'Adult Acute Lymphoblastic Leukemia', 'Chronic Myeloproliferative Neoplasms', 'Parathyroid Cancer', 'Lip and Oral Cavity Cancer', 'Childhood Craniopharyngioma', 'Childhood Ependymoma', 'Childhood Central Nervous System Germ Cell Tumors', 'Ovarian Germ Cell Tumors', 'Childhood Hodgkin Lymphoma', 'Penile Cancer', 'Oropharyngeal Cancer', 'Male Breast Cancer', 'Primary Myelofibrosis', 'Urethral Cancer', 'Melanoma', 'Childhood Central Nervous System Atypical Teratoid/Rhabdoid Tumor', 'Retinoblastoma', 'Childhood Acute Lymphoblastic Leukemia', 'Childhood Liver Cancer', 'Mycosis Fungoides and the Szary Syndrome', 'Ewing Sarcoma', 'Childhood Brain and Spinal Cord Tumors', 'Langerhans Cell Histiocytosis', 'Paranasal Sinus and Nasal Cavity Cancer', 'Campylobacter Infections', 'Pregnancy and Substance Abuse', 'Radiation Therapy', 'Rehabilitation', 'Iron', 'Pregnancy and Nutrition', 'Interstitial Lung Diseases', 'Volcanoes', 'Hiccups', 'Premenstrual Syndrome', 'Dizziness and Vertigo', 'Oxygen Therapy', "Huntington's Disease", 'Premature Babies', 'Mouth Disorders', 'Vision Impairment and Blindness', 'Tremor', 'Nose Injuries and Disorders', 'Diphtheria', 'Esophageal Cancer', 'HIV/AIDS Medicines', 'Rural Health Concerns', 'Skin Conditions', 'Penis Disorders', 'Mumps', 'Syphilis', 'Club Drugs', 'Newborn Screening', 'Emphysema', 'Surgery', 'Ulcerative Colitis', 'Gallstones', 'Childbirth Problems', 'Palliative Care', 'Financial Assistance', 'Amyloidosis', 'Tubal Ligation', 'Medicines and Children', 'Fungal Infections', 'Rashes', 'Endocarditis', 'Ultrasound', 'Bedwetting', 'Hip Injuries and Disorders', 'Nutrition', 'Hodgkin Disease', 'Pemphigus', 'Endometriosis', 'Fatigue', 'Speech and Communication Disorders', 'Frostbite', 'Heroin', 'International Health', 'Heart Valve Diseases', 'Diabetic Nerve Problems', 'Blood Count Tests', 'Von Hippel-Lindau Disease', 'Organ Donation', 'Alpha-1 Antitrypsin Deficiency', 'Pneumonia', 'Hepatitis B', 'Noise', 'Hiatal Hernia', 'Sprains and Strains', 'Bowel Movement', 'Coma', 'Choking', 'Dwarfism', 'Post-Traumatic Stress Disorder', 'Sodium', 'Lactose Intolerance', 'Vitiligo', 'Methamphetamine', 'Native American Health', 'Prescription Drug Abuse', 'Cold and Cough Medicines', 'Canker Sores', 'Measles', 'Stillbirth', 'Cocaine', 'Metabolic Syndrome', 'Hemophilia', 'Bursitis', 'Cerebellar Disorders', 'Water Pollution', 'Bacterial Infections', 'Clostridium Difficile Infections', 'Ozone', 'Erectile Dysfunction', 'Chemical Emergencies', 'Tendinitis', 'Cat Scratch Disease', 'Weight Control', "Bell's Palsy", 'Jaundice', 'Aneurysms', 'MRI Scans', 'Lead Poisoning', 'Bleeding Disorders', 'Parasitic Diseases', 'Coronavirus Infections', 'Pulmonary Fibrosis', 'Impaired Driving', 'Sun Exposure', 'Pregnancy and Medicines', 'Metabolic Panel', 'Aplastic Anemia', 'Tornadoes', 'Attention Deficit Hyperactivity Disorder', 'Mood Disorders', 'Degenerative Nerve Diseases', 'Granulomatosis with Polyangiitis', 'Genetic Disorders', 'Health Disparities', 'Fever', 'Abdominal Pain', 'Transient Ischemic Attack', 'Germs and Hygiene', 'Aortic Aneurysm', 'Soft Tissue Sarcoma', 'Infections and Pregnancy', 'Eczema', 'Sinusitis', 'Scars', 'Infant and Newborn Development', 'Exercise and Physical Fitness', 'Heart Diseases', 'Neurologic Diseases', 'Rotator Cuff Injuries', 'Liver Cancer', 'Haemophilus Infections', 'Autism Spectrum Disorder', 'Heel Injuries and Disorders', 'Arthritis', "Meniere's Disease", 'Bereavement', 'Smokeless Tobacco', 'Head Injuries', 'Child Behavior Disorders', 'Deep Vein Thrombosis', 'Mycobacterial Infections', 'Health Problems in Pregnancy', 'Stem Cells', 'Respiratory Failure', 'Diabetic Kidney Problems', 'Plastic and Cosmetic Surgery', 'Acne', 'Cirrhosis', 'Endocrine Diseases', 'Emergency Medical Services', 'DASH Diet', 'Common Infant and Newborn Problems', 'Sexual Assault', 'Carbohydrates', 'Nutritional Support', 'Child Safety', 'Teen Health', 'Autonomic Nervous System Disorders', 'Fires', 'Hantavirus Infections', 'Eating Disorders', 'Foreign Bodies', 'Kidney Failure', 'Toddler Development', 'Breathing Problems', 'Biodefense and Bioterrorism', 'Adrenal Gland Disorders', 'Antioxidants', 'Body Weight', 'Antibiotics', 'Steroids', 'Hemorrhagic Stroke', 'Tailbone Disorders', 'Mitral Valve Prolapse', 'Motion Sickness', 'Spinal Stenosis', 'Motor Vehicle Safety', 'Healthy Aging', 'Colonoscopy', 'Ovarian Cysts', 'Mercury', 'Phobias', 'Diarrhea', 'Cochlear Implants', 'Nutrition for Seniors', 'Infectious Arthritis', 'Malnutrition', 'Food Allergy', 'Tinnitus', 'B Vitamins', 'Pain Relievers', 'Leg Injuries and Disorders', 'Uterine Cancer', 'Hemorrhoids', 'Wilson Disease', 'Tick Bites', 'Speech and Language Problems in Children', 'Chiropractic', 'Safety', 'Secondhand Smoke', 'Diets', 'Diabetic Eye Problems', 'Poison Ivy, Oak and Sumac', 'Animal Bites', 'Mammography', 'Health Screening', 'Acoustic Neuroma', 'Arm Injuries and Disorders', 'Vegetarian Diet', 'Small Intestine Disorders', 'Asian American Health', 'Asthma', 'Sleep Disorders', 'Turner Syndrome', 'Shock', 'Artificial Limbs', 'Diabetic Diet', 'Pesticides', 'Hidradenitis Suppurativa', 'Antidepressants', 'Restless Legs', 'Eosinophilic Disorders', 'Wilms Tumor', 'Hazardous Waste', 'Botox', 'Medication Errors', 'Moles', 'Swallowing Disorders', 'Tonsils and Adenoids', 'Compulsive Gambling', 'Coping with Chronic Illness', 'Minerals', 'Botulism', 'Hypoglycemia', 'Stuttering', 'Gun Safety', 'Movement Disorders', "Crohn's Disease", 'Bullying', 'Diabetes Type 2', 'Hyperglycemia', 'Native Hawaiian and Pacific Islander Health', 'Acupuncture', 'Vitamin D', 'Liver Function Tests', 'Underage Drinking', 'Chiari Malformation', 'Prediabetes', 'Lymphedema', 'Dietary Fats', 'Leukodystrophies', 'Biopsy', 'Seasonal Affective Disorder', 'Caregivers', 'Intestinal Cancer', 'Dystonia', 'Macular Degeneration', 'Pacemakers and Implantable Defibrillators', 'Male Infertility', 'Medicare Prescription Drug Coverage', 'Household Products', 'Floods', 'Disabilities', 'Childhood Brain Tumors', 'Esophagus Disorders', 'Pulmonary Hypertension', "Men's Health", 'Chickenpox', 'Hearing Disorders and Deafness', 'Hospice Care', 'Eye Wear', 'Temporomandibular Joint Dysfunction', 'Indoor Air Pollution', 'Polymyalgia Rheumatica', 'Understanding Medical Research', 'Alcoholism and Alcohol Abuse', 'Nausea and Vomiting', 'Pleural Disorders', 'Patient Safety', 'Vitamin C', 'Poisoning', 'Constipation', 'Cholesterol', 'HIV/AIDS in Women', 'Carpal Tunnel Syndrome', 'Air Pollution', 'Tetanus, Diphtheria, and Pertussis Vaccines', 'Dementia', 'Alcohol', 'Polycystic Ovary Syndrome', 'Pancreas Transplantation', 'Anxiety', 'Birth Defects', 'Spina Bifida', 'Blood Disorders', 'Nail Diseases', 'Spinal Cord Diseases', 'Sepsis', 'Sexual Problems in Men', 'After Surgery', 'Fetal Health and Development', 'Cerebral Palsy', 'Progressive Supranuclear Palsy', 'Water Safety (Recreational)', 'Bronchial Disorders', 'Lymphoma', 'Pituitary Disorders', 'Coronary Artery Bypass Surgery', 'Bipolar Disorder', 'Parathyroid Disorders', 'Fractures', 'Sciatica', 'Urinalysis', 'Helicobacter Pylori Infections', 'Infectious Diseases', "Women's Health Checkup", 'Aphasia', 'Paralysis', 'Lewy Body Disease', 'Angioplasty', 'Laboratory Tests', 'Uterine Diseases', 'Elder Abuse', 'Teenage Pregnancy', 'Vascular Diseases', 'Blood Clots', 'Dentures', 'Inhalation Injuries', 'Prader-Willi Syndrome', 'Lice', 'Indigestion', 'Diagnostic Imaging', 'Scleroderma', 'Shoulder Injuries and Disorders', 'Cardiac Rehabilitation', 'Giant Cell Arteritis', 'Blood Thinners', 'Circumcision', 'Infant and Newborn Care', 'Hay Fever', 'Kidney Cancer', 'Cervical Cancer', 'Islet Cell Transplantation', 'Hemochromatosis', 'Viral Infections', 'Laser Eye Surgery', 'Urine and Urination', "Klinefelter's Syndrome", 'Schizophrenia', 'Tooth Decay', 'Premature Ovarian Failure', 'Thalassemia', 'Breast Reconstruction', 'Joint Disorders', 'Bleeding', 'Parenting', "Traveler's Health", 'Bowel Incontinence', 'Hysterectomy', 'Staphylococcal Infections', 'Cancer Chemotherapy', 'Abscess', 'Aspergillosis', 'Breast Diseases', 'Rare Diseases', 'Rubella', 'Anaphylaxis', 'Connective Tissue Disorders', 'Down Syndrome', 'Immune System and Disorders', "Behcet's Syndrome", 'Hepatitis', 'Marijuana', 'Genetic Testing', 'Ureteral Disorders', 'Diabetic Heart Disease', 'Porphyria', 'Common Cold', 'Pancreatic Diseases', 'Inhalants', 'Ehlers-Danlos Syndrome', 'MRSA', 'Listeria Infections', 'Ankylosing Spondylitis', 'HIV/AIDS', 'Limb Loss', 'Cervix Disorders', 'Hair Problems', 'Breastfeeding', 'Farm Health and Safety', 'Ovarian Disorders', 'Clinical Trials', 'Flu Shot', 'Toxoplasmosis', 'Tumors and Pregnancy', "Kaposi's Sarcoma", 'Adhesions', 'Bone Marrow Diseases', 'Urethral Disorders', 'High Blood Pressure in Pregnancy', 'Patient Rights', 'Genes and Gene Therapy', "Seniors' Health", 'A1C', 'Menopause', 'Scabies', 'Medicare', 'Hormone Replacement Therapy', 'Blood Pressure Medicines', 'CT Scans', 'Talking With Your Doctor', 'Prostate Cancer Screening', 'Mobility Aids', 'Rett Syndrome', 'Genital Herpes', 'Neurofibromatosis', 'Multiple Myeloma', 'Child Nutrition', 'Cardiomyopathy', 'Tuberous Sclerosis', 'Walking Problems', 'Blood Sugar', 'Fetal Alcohol Spectrum Disorders', 'Bone Marrow Transplantation', 'Diabetes Medicines', 'Hormones', 'Fistulas', 'Ear Infections', 'Pressure Sores', 'Mastectomy', 'Advance Directives', 'Fire Safety', 'Chikungunya', 'Testicular Disorders', 'Neck Injuries and Disorders', 'Over-the-Counter Medicines', 'Hepatitis C', 'Ovarian Cancer', 'Pheochromocytoma', 'Sports Safety', 'Phenylketonuria', 'Cosmetic Dentistry', 'Health Literacy', 'Adrenal Gland Cancer', 'Bone Density', 'Radiation Emergencies', 'Sports Injuries', 'Medicines', 'Childbirth', 'Peritoneal Disorders', 'Arteriovenous Malformations', 'Bone Diseases', "Legionnaires' Disease", 'Rabies', 'Dialysis', 'Reproductive Hazards', 'Coronary Artery Disease', 'Tongue Disorders', 'Child Sexual Abuse', 'Toddler Health', 'Complex Regional Pain Syndrome', 'Acute Bronchitis', 'Pinkeye', 'Dislocations', 'Mental Health', 'Migraine', 'Blood', 'Bad Breath', 'Medical Device Safety', 'Zika Virus', 'Sports Fitness', 'Bladder Cancer', 'Cryptosporidiosis', 'Bedbugs', 'Hip Replacement', 'Dengue', 'Peripheral Nerve Disorders', 'Fibromyalgia', 'Guillain-Barre Syndrome', 'Corns and Calluses', 'Fragile X Syndrome', 'Vaginal Bleeding', 'Foot Injuries and Disorders', 'Pelvic Inflammatory Disease', "Children's Health", 'Herpes Simplex', 'Cartilage Disorders', 'Lymphatic Diseases', 'Irritable Bowel Syndrome', 'Smallpox', 'Spine Injuries and Disorders', 'Ebola', 'Mental Disorders', 'Chagas Disease', 'GERD', 'Quitting Smoking', 'Prenatal Testing', 'Voice Disorders', 'Benign Tumors', 'Rosacea', 'Eye Infections', 'Myasthenia Gravis', 'School Health', 'Enlarged Prostate (BPH)', 'Stomach Disorders', 'Itching', 'Healthy Living', 'Mesothelioma', 'Bile Duct Diseases', 'Fluid and Electrolyte Balance', 'Infection Control', 'Potassium', 'Warts', 'Heart Transplantation', 'Delirium', 'Prenatal Care', 'Throat Disorders', 'Child Development', 'Pinworms', 'Skin Pigmentation Disorders', 'Health Insurance', 'Climate Change', 'Amblyopia', 'Immunization', 'Cancer', 'Refractive Errors', 'Cesarean Section', 'Digestive Diseases', 'Acute Myeloid Leukemia', 'Critical Care', 'Tooth Disorders', 'Liver Diseases', 'Genetic Brain Disorders', 'Sarcoidosis', 'Trigeminal Neuralgia', 'Neuromuscular Disorders', 'Snoring', 'Whooping Cough', 'Varicose Veins', 'Chronic Myeloid Leukemia', 'Statins', 'Scoliosis', 'Retinal Disorders', 'Monkeypox Virus Infections', 'Thymus Cancer', 'Hemorrhagic Fevers', 'Hand Injuries and Disorders', 'College Health', "Raynaud's Disease", 'Cytomegalovirus Infections', 'Gangrene', 'Sudden Infant Death Syndrome', 'Brain Aneurysm', 'Corneal Disorders', 'Winter Weather Emergencies', 'Headache', 'E-Cigarettes', 'Eye Movement Disorders', 'Croup', 'Assistive Devices', "Alzheimer's Caregivers", "Friedreich's Ataxia", 'Childhood Immunization', 'Wrist Injuries and Disorders', 'Dietary Fiber', 'Colonic Polyps', 'Drug Safety', 'Thyroid Cancer', 'Dietary Supplements', 'Veterans and Military Health', 'Ischemic Stroke', 'Twins, Triplets, Multiple Births', 'Vasectomy', 'Living with HIV/AIDS', 'Impetigo', 'End of Life Issues', 'Osteogenesis Imperfecta', 'Managed Care', 'Cancer in Children', 'Appendicitis', 'Allergy', 'Medicaid', 'Stomach Cancer', 'Falls', 'Infectious Mononucleosis', 'Hair Loss', 'Veterans and Military Family Health', 'Domestic Violence', 'Sexual Problems in Women', 'Stress', 'Ankle Injuries and Disorders', 'Latex Allergy', 'Cough', 'Postpartum Depression', 'Health Occupations', 'Retinal Detachment', 'Oil Spills', 'Sleep Apnea', 'Gallbladder Diseases', 'Prostate Diseases', 'Animal Diseases and Your Health', 'Platelet Disorders', 'Nursing Homes', 'Pet Health', 'Tuberculosis', 'Mild Cognitive Impairment', 'Myositis', "Athlete's Foot", 'Hepatitis A', 'Tourette Syndrome', 'Pericardial Disorders', 'Antibiotic Resistance', 'HPV', 'Homeless Health Concerns', 'Anthrax', 'Developmental Disabilities', 'Creatinine', 'Head Lice', 'Chest Injuries and Disorders', 'Cystic Fibrosis', 'Obesity', 'Nasal Cancer', 'Burns', "Children's Page", 'Female Infertility', 'Radiation Exposure', 'Thyroid Diseases', 'Heart Diseases--Prevention', 'Charcot-Marie-Tooth Disease', 'Cloning', 'Atrial Fibrillation', 'Vitamin K', 'Piercing and Tattoos', 'Traumatic Brain Injury', 'Internet Safety', 'Birthmarks', 'Chest Pain', 'Marfan Syndrome', 'Carotid Artery Disease', 'Brachial Plexus Injuries', 'Usher Syndrome', 'Creutzfeldt-Jakob Disease', 'Leishmaniasis', 'Facial Injuries and Disorders', 'Health Checkup', 'Edema', 'Pulmonary Embolism', 'E. Coli Infections', 'Preterm Labor', 'Elbow Injuries and Disorders', 'Teen Violence', 'Interstitial Cystitis', 'Reye Syndrome', 'Asthma in Children', 'Rh Incompatibility', 'Seizures', 'Back Injuries', 'Miscarriage', 'Asbestos', 'Infant and Newborn Nutrition', 'Cervical Cancer Screening', 'Anesthesia', 'Child Dental Health', 'Ostomy', 'Osteonecrosis', 'Tears', 'Anal Disorders', 'Diabetes Type 1', 'Spinal Cord Injuries', 'Yeast Infections', 'Diabetes in Children and Teens', 'Chronic Pain', 'Cleft Lip and Palate', 'Teen Development', 'Home Care Services', 'Dietary Proteins', 'Stroke Rehabilitation', 'Skin Aging', 'Fifth Disease', 'Skin Infections', 'Health Fraud', 'Oral Cancer', 'Lupus', 'Peptic Ulcer', 'Occupational Health for Healthcare Providers', 'Exercise for Seniors', 'Hearing Problems in Children', 'Plague', 'Meningitis', 'Norovirus Infections', 'Vitamin E', 'Drowning', "Sjogren's Syndrome", 'Organ Transplantation', 'Foot Health', 'Epilepsy', 'Neural Tube Defects', 'Divorce', 'Streptococcal Infections', 'Gastrointestinal Bleeding', 'Cholera', 'Earthquakes', 'Arrhythmia', 'Triglycerides', 'Pelvic Pain', 'Radon', 'Ear Disorders', 'Molds', 'Foodborne Illness', 'First Aid', 'Vitamins', 'Brain Diseases', 'Tracheal Disorders', 'Choosing a Doctor or Health Care Service', 'Sexual Health', 'Diabetic Foot', 'Hurricanes', 'Head and Neck Cancer', "Women's Health", 'HIV/AIDS and Pregnancy', 'Postpartum Care', 'Ectopic Pregnancy', 'Addison Disease', 'West Nile Virus', 'Cardiac Arrest', 'Cancer--Living with Cancer', 'Sore Throat', 'Child Care', 'Finger Injuries and Disorders', 'Panic Disorder', 'Jaw Injuries and Disorders', 'Arsenic', 'Collapsed Lung', 'Medical Ethics', 'Pneumococcal Infections', 'Drugs and Young People', 'Respiratory Syncytial Virus Infections', 'Coping with Disasters', 'Low Blood Pressure', 'Muscle Cramps', 'Lung Diseases', 'Trichomoniasis', 'Vulvar Disorders', 'Cancer Alternative Therapies', 'Toddler Nutrition', 'Exercise for Children', 'Memory', 'Bladder Diseases', 'Valley Fever', 'Malabsorption Syndromes', 'Health Statistics', 'Acute Lymphocytic Leukemia', 'Multiple Sclerosis', 'Intestinal Obstruction', 'Cellulitis', 'Overactive Bladder', 'Celiac Disease', 'Preconception Care', 'Color Blindness', 'Knee Injuries and Disorders', 'Food Safety', 'Back Pain', 'Throat Cancer', 'Chronic Kidney Disease', 'Vital Signs', 'Diabetes Complications', 'Occupational Health', 'Personal Health Records', 'Lung Transplantation', 'Muscle Disorders', 'Diabetes and Pregnancy', 'Spinal Muscular Atrophy', 'Eyelid Disorders', 'Birth Control', 'Genital Warts', 'Dual Diagnosis', 'Angina', 'Barotrauma', 'Cold Sores', 'Electromagnetic Fields', 'Gay, Lesbian, Bisexual and Transgender Health', 'Lyme Disease', 'Adoption', 'Sweat', 'Health Facilities', 'Giardia Infections', 'Brain Malformations', 'Spleen Diseases', 'Assisted Living', 'Wounds and Injuries', 'Hearing Aids', 'Genetic Counseling', 'Kawasaki Disease', 'Obesity in Children', 'HIV/AIDS and Infections', 'Insect Bites and Stings', 'Cosmetics', 'Bile Duct Cancer', 'Smoking and Youth', 'Hyperthyroidism', 'Gas', 'Smoking', 'Pelvic Support Problems', 'Spider Bites', 'Endoscopy', 'Toe Injuries and Disorders', 'Flu', 'Family History', 'Caffeine', 'Evaluating Health Information', 'Sickle Cell Anemia', 'Teen Mental Health', 'Personality Disorders', 'Hypothyroidism', 'Hernia', 'Uncommon Infant and Newborn Problems', 'X-Rays', 'Kidney Stones', 'Dental Health', 'Colonic Diseases', 'Heart Disease in Women', 'Vaginal Cancer', 'Dehydration', 'Disaster Preparation and Recovery', 'Encephalitis', 'Hives', 'Anatomy', 'Suicide', 'Assisted Reproductive Technology', 'Autoimmune Diseases', 'Carbon Monoxide Poisoning', 'Craniofacial Abnormalities', 'Heartburn', 'Tay-Sachs Disease', 'Vitamin A', 'CPR', 'Malaria', 'Juvenile Arthritis', "Cushing's Syndrome", 'Puberty', 'Liver Transplantation', 'Polio and Post-Polio Syndrome', 'Growth Disorders', 'Learning Disorders', 'G6PD Deficiency', 'Bruises', 'Toilet Training', 'Chronic Fatigue Syndrome', 'Blood Transfusion and Donation', 'Pregnancy', 'Herniated Disk', 'Ergonomics', 'Gum Disease', 'Menstruation', 'Child Mental Health', 'Brain Tumors', 'Psoriatic Arthritis', "Teens' Page", 'Tetanus'

---

# 🏥 MedDoc Integration Summary

## How This QA Pipeline Powers MedDoc

This notebook demonstrates the core NLP engine that underpins MedDoc's intelligent medical assistant. Here's how each component maps to MedDoc features:

| Notebook Component | MedDoc Feature | Integration Point |
|---|---|---|
| BERT QA Model | AI Assistant Symptom Triage | `patient.assistant.tsx` |
| MedQuAD Dataset | Medical Knowledge Base | `ai.service.ts` |
| Focus Area Mapping | Follow-up Question Generation | Adaptive interview flow |
| Interactive QA Loop | Real-time Patient Interaction | Chat-based symptom collection |

## Key Enhancements for MedDoc

1. **Enhanced Dataset Coverage**: MedDoc augments the MedQuAD dataset with `Final_Augmented_dataset_Diseases_and_Symptoms.csv`, expanding coverage to 200+ conditions with symptom-severity mapping.

2. **Adaptive Follow-up Questions**: Unlike static QA, MedDoc's assistant uses the BERT confidence scores to determine when additional follow-up questions are needed for accurate triage.

3. **Multi-Modal Integration**: The QA engine works alongside MedDoc's medical image analysis and report parsing pipelines to provide a comprehensive diagnostic assessment.

4. **Clinical Safety**: All AI-generated responses include appropriate medical disclaimers and recommendations to consult healthcare professionals.

> ⚠️ **Disclaimer**: MedDoc's AI Assistant is designed for preliminary health assessment and should not replace professional medical advice, diagnosis, or treatment.
